In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import seaborn as sns
import scipy.stats as stats

from Bio.Seq import reverse_complement
import Bio.SeqUtils as Seq

In [3]:
def get_dict_of_seq(fasta_file):
    """ function that converts a fasta file to a dictionnary of sequences
    fasta_file: the input fasta file
    """
    
    file_fasta_dict = {}
    # output dict of imported seqs
    
    with open(fasta_file, 'r') as fasta:    
        for line in fasta:
            # loops through the file

            if line.startswith('>') == True:
                seq_info = line.strip('>').strip('\n').split('\t')[0]
                file_fasta_dict[seq_info] = ''
                # checks if seq header

            else:
                file_fasta_dict[seq_info] += line.strip('\n')
                # If not, append nt to current seq
                
    return file_fasta_dict

In [6]:
sgRNA_df = pd.read_csv('./sgRNA_df.csv')


In [7]:
sgRNA_df['control'] = sgRNA_df['Gene'].str.count('R-sgRNA')
# will be 1 for random sgRNAs
sgRNA_df.at[552, 'control']=2
#will be 2 for the backbone
sgRNA_df

,Gene,SgRNA,Barcode,control
0,C100220W,1,TTGGCTTTCCTCCCCTTAAC,0
1,C100220W,2,TCCTTGAATTTCTTGGTAGA,0
2,C100220W,3,CGATGATTCCTTGAATTTCT,0
3,C100220W,4,CAACAGTAGCAGAATTGTAC,0
4,C100400W,5,TTAATGTCGCTTAGTGATGG,0
...,...,...,...,...
548,R-sgRNA 27,551,GGTATCGTCGTGTCTCATTA,1
549,R-sgRNA 28,552,CGCGTCCAGGTTCTTCTGGA,1
550,R-sgRNA 29,553,TGTTGGATCGTCCCTAGGAA,1
551,R-sgRNA 30,554,TCTGACGATCTGTTGTGAGA,1


In [8]:
sgRNA_df['Gene'] = sgRNA_df.apply(lambda x: x.Gene[0:2]+'_'+x.Gene[2:] if len(x.Gene) <=9 else x.Gene, axis=1)

sgRNA_df

,Gene,SgRNA,Barcode,control
0,C1_00220W,1,TTGGCTTTCCTCCCCTTAAC,0
1,C1_00220W,2,TCCTTGAATTTCTTGGTAGA,0
2,C1_00220W,3,CGATGATTCCTTGAATTTCT,0
3,C1_00220W,4,CAACAGTAGCAGAATTGTAC,0
4,C1_00400W,5,TTAATGTCGCTTAGTGATGG,0
...,...,...,...,...
548,R-sgRNA 27,551,GGTATCGTCGTGTCTCATTA,1
549,R-sgRNA 28,552,CGCGTCCAGGTTCTTCTGGA,1
550,R-sgRNA 29,553,TGTTGGATCGTCCCTAGGAA,1
551,R-sgRNA 30,554,TCTGACGATCTGTTGTGAGA,1


In [9]:
ca_orf_dict = get_dict_of_seq('./C_albicans_SC5314_A22_current_orf_genomic_1000.fasta')

sys_name_to_alias_dict = {}
seq_by_alias_dict = {}

for keys in list(ca_orf_dict.keys()):    
    gene_info = keys.split(' ')
    sys_name = gene_info[0][:-2]
    alias = gene_info[1]
    
    if '_B' not in alias: #  or '_B' in alias
        
        if '_A' in alias:
            alias = alias[:-2]
        #print(sys_name, alias)

        if sys_name in list(sys_name_to_alias_dict.keys()):
            if len(alias) < len(sys_name_to_alias_dict[sys_name]):
                sys_name_to_alias_dict[sys_name]=alias

                seq_by_alias_dict[alias] = ca_orf_dict[keys]

        else:       
            sys_name_to_alias_dict[sys_name] = alias
            seq_by_alias_dict[alias] = ca_orf_dict[keys]
    


In [10]:
sgRNA_df['alias'] = sgRNA_df.apply(lambda x: sys_name_to_alias_dict[x.Gene] if len(x.Gene) <=9 else np.NaN, axis=1)
sgRNA_df

,Gene,SgRNA,Barcode,control,alias
0,C1_00220W,1,TTGGCTTTCCTCCCCTTAAC,0,PHR2
1,C1_00220W,2,TCCTTGAATTTCTTGGTAGA,0,PHR2
2,C1_00220W,3,CGATGATTCCTTGAATTTCT,0,PHR2
3,C1_00220W,4,CAACAGTAGCAGAATTGTAC,0,PHR2
4,C1_00400W,5,TTAATGTCGCTTAGTGATGG,0,SVF1
...,...,...,...,...,...
548,R-sgRNA 27,551,GGTATCGTCGTGTCTCATTA,1,NaN
549,R-sgRNA 28,552,CGCGTCCAGGTTCTTCTGGA,1,NaN
550,R-sgRNA 29,553,TGTTGGATCGTCCCTAGGAA,1,NaN
551,R-sgRNA 30,554,TCTGACGATCTGTTGTGAGA,1,NaN


In [11]:
def get_sgrna_loc_strand_pam(index, df, alias_col, barcode_col, ref_req_dict):
    alias = df.loc[index][alias_col]
    sgrna_seq =  df.loc[index][barcode_col]
    
    strand = np.NaN
    atg_dist = np.NaN
    PAM = np.NaN
    
    if type(alias)==float:
        return np.NaN, np.NaN, np.NaN
    
    else:
        ref_seq = ref_req_dict[alias]
        
        if sgrna_seq in ref_seq:
            
            strand = '+'
            atg_dist = 1000-(ref_seq.index(sgrna_seq)+10)
            PAM = ref_seq[(ref_seq.index(sgrna_seq)+20):(ref_seq.index(sgrna_seq)+23)]
            
        elif reverse_complement(sgrna_seq) in ref_seq:
            
            strand = '-'
            atg_dist = 1000-(ref_seq.index(reverse_complement(sgrna_seq))+10)
            PAM = reverse_complement(ref_seq[ref_seq.index(reverse_complement(sgrna_seq))-3:ref_seq.index(reverse_complement(sgrna_seq))])
            
        else:
            print(index)
            
    return strand, atg_dist, PAM

get_sgrna_loc_strand_pam(91, sgRNA_df, 'alias', 'Barcode', seq_by_alias_dict)

('-', 47, 'TGG')

In [12]:
PAM_dict = {}
strand_dict = {}
atg_dist_dict = {}
B_mismatch_dict = {}

B_mismatch_list  = [148, 149, 151, 347, 364, 477, 481, 493]

for index in list(sgRNA_df.index):

    strand, atg_dist, PAM = get_sgrna_loc_strand_pam(index, sgRNA_df, 'alias', 'Barcode', seq_by_alias_dict)

    PAM_dict[index] = PAM
    strand_dict[index] = strand
    atg_dist_dict[index] = atg_dist
    
    if index in B_mismatch_list:
        B_mismatch_dict[index] = 'yes'
        
    else:
        B_mismatch_dict[index] = 'no'

sgRNA_df['PAM'] = pd.Series(PAM_dict)
sgRNA_df['strand'] = pd.Series(strand_dict)
sgRNA_df['atg_dist'] = pd.Series(atg_dist_dict)
sgRNA_df['B_mismatch'] = pd.Series(B_mismatch_dict)

sgRNA_df

,Gene,SgRNA,Barcode,control,alias,PAM,strand,atg_dist,B_mismatch
0,C1_00220W,1,TTGGCTTTCCTCCCCTTAAC,0,PHR2,TGG,+,112.0,no
1,C1_00220W,2,TCCTTGAATTTCTTGGTAGA,0,PHR2,AGG,-,45.0,no
2,C1_00220W,3,CGATGATTCCTTGAATTTCT,0,PHR2,TGG,-,38.0,no
3,C1_00220W,4,CAACAGTAGCAGAATTGTAC,0,PHR2,AGG,+,288.0,no
4,C1_00400W,5,TTAATGTCGCTTAGTGATGG,0,SVF1,AGG,-,177.0,no
...,...,...,...,...,...,...,...,...,...
548,R-sgRNA 27,551,GGTATCGTCGTGTCTCATTA,1,NaN,NaN,NaN,NaN,no
549,R-sgRNA 28,552,CGCGTCCAGGTTCTTCTGGA,1,NaN,NaN,NaN,NaN,no
550,R-sgRNA 29,553,TGTTGGATCGTCCCTAGGAA,1,NaN,NaN,NaN,NaN,no
551,R-sgRNA 30,554,TCTGACGATCTGTTGTGAGA,1,NaN,NaN,NaN,NaN,no


In [13]:
tss_annot_df = pd.read_excel('./TSS_annot.xlsx')
tss_annot_df

,Gene_ID,Weighted_5_UTR_Length,TPM*,TSS Data ID
0,C100010WA,0,NaN,NaN
1,C100020CA,486,604.413805,CAALFM_C100020CA
2,C100030CA,122,6.580078,CAALFM_C100030CA
3,C100040WA,0,NaN,NaN
4,C100050CA,0,NaN,NaN
...,...,...,...,...
6662,CM00480C,0,NaN,NaN
6663,CM00490C,0,NaN,NaN
6664,CM00500C,0,NaN,NaN
6665,CM00510W,0,NaN,NaN


In [14]:
tss_annot_df['Gene_rename'] = tss_annot_df.apply(lambda x: x.Gene_ID[0:2]+'_'+x.Gene_ID[2:8], axis=1)

tss_annot_df

,Gene_ID,Weighted_5_UTR_Length,TPM*,TSS Data ID,Gene_rename
0,C100010WA,0,NaN,NaN,C1_00010W
1,C100020CA,486,604.413805,CAALFM_C100020CA,C1_00020C
2,C100030CA,122,6.580078,CAALFM_C100030CA,C1_00030C
3,C100040WA,0,NaN,NaN,C1_00040W
4,C100050CA,0,NaN,NaN,C1_00050C
...,...,...,...,...,...
6662,CM00480C,0,NaN,NaN,CM_00480C
6663,CM00490C,0,NaN,NaN,CM_00490C
6664,CM00500C,0,NaN,NaN,CM_00500C
6665,CM00510W,0,NaN,NaN,CM_00510W


In [15]:
tss_annot_df['TPM*']

0              NaN
1       604.413805
2         6.580078
3              NaN
4              NaN
           ...    
6662           NaN
6663           NaN
6664           NaN
6665           NaN
6666           NaN
Name: TPM*, Length: 6667, dtype: float64

In [16]:
tss_annot_df.dropna(axis=0, subset=['TPM*'], inplace=True)
tss_annot_df

,Gene_ID,Weighted_5_UTR_Length,TPM*,TSS Data ID,Gene_rename
1,C100020CA,486,604.413805,CAALFM_C100020CA,C1_00020C
2,C100030CA,122,6.580078,CAALFM_C100030CA,C1_00030C
5,C100060WA,14,905.396887,TUP1,C1_00060W
6,C100070WA,10,95.681371,MVD,C1_00070W
10,C100110WA,12,64.370356,CCT8,C1_00110W
...,...,...,...,...,...
6609,CR10810CA,12,34.426220,CAALFM_CR10810CA,CR_10810C
6610,CR10820WA,10,27.957397,CAALFM_CR10820WA,CR_10820W
6611,CR10830CA,18,14.749551,CAALFM_CR10830CA,CR_10830C
6612,CR10840CA,4,2228.692491,XYL2,CR_10840C


In [17]:
dist_TSS_dict = {}
abs_TSS_dist_dict = {}

w_annot = list(tss_annot_df['Gene_rename'])

annot_match = 0

for index in list(sgRNA_df.index):
    
    gene_id = sgRNA_df.at[index, 'Gene']
    sgrna_pos = sgRNA_df.at[index, 'atg_dist']
    
    if gene_id in w_annot:
        annot_match+=1
        
        annot_slice = tss_annot_df[tss_annot_df['Gene_rename'] == gene_id]
        TSS = float(annot_slice['Weighted_5_UTR_Length'])
        
        dist_TSS_dict[index] = TSS-sgrna_pos
        abs_TSS_dist_dict[index] = abs(TSS-sgrna_pos)
        
    else:
        dist_TSS_dict[index] = np.NaN
        abs_TSS_dist_dict[index] = np.NaN
                
    
sgRNA_df['dist_TSS'] = pd.Series(dist_TSS_dict)
sgRNA_df['abs_TSS_dist'] = pd.Series(abs_TSS_dist_dict)

/tmp/ipykernel_12745/1763379933.py:17: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  TSS = float(annot_slice['Weighted_5_UTR_Length'])


In [18]:
sgRNA_df

,Gene,SgRNA,Barcode,control,alias,PAM,strand,atg_dist,B_mismatch,dist_TSS,abs_TSS_dist
0,C1_00220W,1,TTGGCTTTCCTCCCCTTAAC,0,PHR2,TGG,+,112.0,no,-9.0,9.0
1,C1_00220W,2,TCCTTGAATTTCTTGGTAGA,0,PHR2,AGG,-,45.0,no,58.0,58.0
2,C1_00220W,3,CGATGATTCCTTGAATTTCT,0,PHR2,TGG,-,38.0,no,65.0,65.0
3,C1_00220W,4,CAACAGTAGCAGAATTGTAC,0,PHR2,AGG,+,288.0,no,-185.0,185.0
4,C1_00400W,5,TTAATGTCGCTTAGTGATGG,0,SVF1,AGG,-,177.0,no,-89.0,89.0
...,...,...,...,...,...,...,...,...,...,...,...
548,R-sgRNA 27,551,GGTATCGTCGTGTCTCATTA,1,NaN,NaN,NaN,NaN,no,NaN,NaN
549,R-sgRNA 28,552,CGCGTCCAGGTTCTTCTGGA,1,NaN,NaN,NaN,NaN,no,NaN,NaN
550,R-sgRNA 29,553,TGTTGGATCGTCCCTAGGAA,1,NaN,NaN,NaN,NaN,no,NaN,NaN
551,R-sgRNA 30,554,TCTGACGATCTGTTGTGAGA,1,NaN,NaN,NaN,NaN,no,NaN,NaN


## GC content

In [21]:
sgRNA_df['GC_content'] = sgRNA_df.apply(lambda x: Seq.gc_fraction(x.Barcode), axis=1)
sgRNA_df

,Gene,SgRNA,Barcode,control,alias,PAM,strand,atg_dist,B_mismatch,dist_TSS,abs_TSS_dist,GC_content
0,C1_00220W,1,TTGGCTTTCCTCCCCTTAAC,0,PHR2,TGG,+,112.0,no,-9.0,9.0,0.500000
1,C1_00220W,2,TCCTTGAATTTCTTGGTAGA,0,PHR2,AGG,-,45.0,no,58.0,58.0,0.350000
2,C1_00220W,3,CGATGATTCCTTGAATTTCT,0,PHR2,TGG,-,38.0,no,65.0,65.0,0.350000
3,C1_00220W,4,CAACAGTAGCAGAATTGTAC,0,PHR2,AGG,+,288.0,no,-185.0,185.0,0.400000
4,C1_00400W,5,TTAATGTCGCTTAGTGATGG,0,SVF1,AGG,-,177.0,no,-89.0,89.0,0.400000
...,...,...,...,...,...,...,...,...,...,...,...,...
548,R-sgRNA 27,551,GGTATCGTCGTGTCTCATTA,1,NaN,NaN,NaN,NaN,no,NaN,NaN,0.450000
549,R-sgRNA 28,552,CGCGTCCAGGTTCTTCTGGA,1,NaN,NaN,NaN,NaN,no,NaN,NaN,0.600000
550,R-sgRNA 29,553,TGTTGGATCGTCCCTAGGAA,1,NaN,NaN,NaN,NaN,no,NaN,NaN,0.500000
551,R-sgRNA 30,554,TCTGACGATCTGTTGTGAGA,1,NaN,NaN,NaN,NaN,no,NaN,NaN,0.450000


In [22]:
list(sgRNA_df.columns)

['Gene',
 'SgRNA',
 'Barcode',
 'control',
 'alias',
 'PAM',
 'strand',
 'atg_dist',
 'B_mismatch',
 'dist_TSS',
 'abs_TSS_dist',
 'GC_content']

## SNPs from WGS

In [23]:
snp_to_sgrna_map = pd.read_csv('./sgRNA_variants_manual_curation.csv', sep=',')
snp_to_sgrna_map

,sgRNA_ID,Chr,position,effect,fRS1,fRS242,fRS246,type,source,comment,Unnamed: 10
0,60,1,838214,True,het,het,B,SC5314_het,het_site_CGD,NaN,NaN
1,68,1,1166862,True,het,het,B,SC5314_het,het_site_CGD,NaN,NaN
2,83,1,1553473,True,het,het,het,SC5314_het,het_site_CGD,NaN,NaN
3,83,1,1553477,True,het,het,het,SC5314_het,het_site_CGD,NaN,NaN
4,85,1,1645351,True,A,A,A,SC5314_het,het_site_CGD,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
122,435,R,201954,True,hom,hom,alt,de_novo,fRS246_intersect,NaN,NaN
123,435,R,201957,True,hom,hom,alt,de_novo,fRS246_intersect,NaN,NaN
124,436,R,201943,True,hom,hom,alt,de_novo,fRS246_intersect,NaN,NaN
125,436,R,201954,True,hom,hom,alt,de_novo,fRS246_intersect,NaN,NaN


In [24]:
snp_w_effect = snp_to_sgrna_map['effect'] == True


In [25]:
frs1_not_altered = snp_to_sgrna_map['fRS1'].isin(['A', 'hom'])
frs242_not_altered = snp_to_sgrna_map['fRS242'].isin(['A', 'hom'])
frs246_not_altered = snp_to_sgrna_map['fRS246'].isin(['A', 'hom'])

In [26]:
frs1_hom_mut = snp_to_sgrna_map['fRS1'].isin(['B', 'alt'])
frs242_hom_mut = snp_to_sgrna_map['fRS242'].isin(['B', 'alt'])
frs246_hom_mut = snp_to_sgrna_map['fRS246'].isin(['B', 'alt'])

In [27]:
n_snps_fRS1 = pd.Series(snp_to_sgrna_map[snp_w_effect&~frs1_not_altered]['sgRNA_ID'].value_counts())
n_snps_fRS242 = pd.Series(snp_to_sgrna_map[snp_w_effect&~frs242_not_altered]['sgRNA_ID'].value_counts())
n_snps_fRS246 = pd.Series(snp_to_sgrna_map[snp_w_effect&~frs246_not_altered]['sgRNA_ID'].value_counts())

n_snps_hom_mut_fRS1 = pd.Series(snp_to_sgrna_map[snp_w_effect&frs1_hom_mut]['sgRNA_ID'].value_counts())
n_snps_hom_mut_fRS242 = pd.Series(snp_to_sgrna_map[snp_w_effect&frs242_hom_mut]['sgRNA_ID'].value_counts())
n_snps_hom_mut_fRS246 = pd.Series(snp_to_sgrna_map[snp_w_effect&frs246_hom_mut]['sgRNA_ID'].value_counts())


n_snps_hom_mut_fRS242

sgRNA_ID
485    3
480    3
133    2
118    1
473    1
135    1
134    1
130    1
129    1
484    1
467    1
119    1
464    1
459    1
458    1
457    1
327    1
326    1
325    1
120    1
136    1
Name: count, dtype: int64

In [28]:
sgRNA_df['snp_fRS1'] = n_snps_fRS1
sgRNA_df['snp_hom_fRS1'] = n_snps_hom_mut_fRS1
sgRNA_df['snp_fRS1'] = sgRNA_df['snp_fRS1'].fillna(0)
sgRNA_df['snp_hom_fRS1'] = sgRNA_df['snp_hom_fRS1'].fillna(0)

sgRNA_df['snp_fRS242'] = n_snps_fRS242
sgRNA_df['snp_hom_fRS242'] = n_snps_hom_mut_fRS242
sgRNA_df['snp_fRS242'] = sgRNA_df['snp_fRS242'].fillna(0)
sgRNA_df['snp_hom_fRS242'] = sgRNA_df['snp_hom_fRS242'].fillna(0)

sgRNA_df['snp_fRS246'] = n_snps_fRS246
sgRNA_df['snp_hom_fRS246'] = n_snps_hom_mut_fRS246
sgRNA_df['snp_fRS246'] = sgRNA_df['snp_fRS246'].fillna(0)
sgRNA_df['snp_hom_fRS246'] = sgRNA_df['snp_hom_fRS246'].fillna(0)



In [29]:
sgRNA_df['snp_hom_fRS242'].value_counts()

snp_hom_fRS242
0.0    532
1.0     18
3.0      2
2.0      1
Name: count, dtype: int64

In [30]:
sgRNA_df[sgRNA_df['snp_hom_fRS242']>0]

,Gene,SgRNA,Barcode,control,alias,PAM,strand,atg_dist,B_mismatch,dist_TSS,abs_TSS_dist,GC_content,snp_fRS1,snp_hom_fRS1,snp_fRS242,snp_hom_fRS242,snp_fRS246,snp_hom_fRS246
118,C1_11500C,119,ACGTTGATTTCGAAATCCCG,0,ARO7,GGG,+,231.0,no,-173.0,173.0,0.45,1.0,0.0,1.0,1.0,1.0,0.0
119,C1_11500C,120,GTTCAATATATACATTACCC,0,ARO7,CGG,-,211.0,no,-153.0,153.0,0.30,1.0,0.0,1.0,1.0,1.0,0.0
120,C1_12260W,121,TCGCATTAGACGAGAGATTC,0,VPH2,AGG,+,108.0,no,-19.0,19.0,0.45,1.0,0.0,1.0,1.0,1.0,0.0
129,C1_12610W,130,AGAAAGAACAAATGAATTTG,0,C1_12610W,AGG,-,117.0,no,-74.0,74.0,0.25,1.0,0.0,1.0,1.0,1.0,0.0
130,C1_12610W,131,GTGATAATGTTATATGTTGT,0,C1_12610W,CGG,-,28.0,no,15.0,15.0,0.25,1.0,0.0,1.0,1.0,1.0,0.0
133,C1_12660W,134,ATTGGATTGAAAGGCGGACG,0,C1_12660W,GGG,-,229.0,no,-137.0,137.0,0.50,2.0,0.0,2.0,2.0,2.0,0.0
134,C1_12660W,135,GATTGGATTGAAAGGCGGAC,0,C1_12660W,GGG,-,228.0,no,-136.0,136.0,0.50,1.0,0.0,1.0,1.0,1.0,0.0
135,C1_12660W,136,AGATTGGATTGAAAGGCGGA,0,C1_12660W,CGG,-,227.0,no,-135.0,135.0,0.45,1.0,0.0,1.0,1.0,1.0,0.0
136,C1_12810W,137,CAAATTGCCTCCCACCACCA,0,RIB7,AGG,+,145.0,no,-70.0,70.0,0.55,1.0,0.0,1.0,1.0,1.0,0.0
325,C5_01720C,328,AAAGTAATATCTTAACGCAA,0,RCY1,TGG,-,158.0,no,59.0,59.0,0.25,1.0,0.0,1.0,1.0,1.0,1.0


In [31]:
n_sgrna_dict = dict(sgRNA_df[sgRNA_df['control']==0]['Gene'].value_counts())

In [32]:
frs1_dist = []
frs242_dist = []
frs246_dist = []

for gene in n_sgrna_dict.keys():

    n_frs1_snp = len(sgRNA_df[(sgRNA_df['Gene']==gene)&(sgRNA_df['snp_fRS1']>0)])
    frs1_dist.append(n_frs1_snp/n_sgrna_dict[gene])

    n_frs242_snp = len(sgRNA_df[(sgRNA_df['Gene']==gene)&(sgRNA_df['snp_fRS242']>0)])
    frs242_dist.append(n_frs242_snp/n_sgrna_dict[gene])

    n_frs246_snp = len(sgRNA_df[(sgRNA_df['Gene']==gene)&(sgRNA_df['snp_fRS246']>0)])
    frs246_dist.append(n_frs246_snp/n_sgrna_dict[gene])
    
    print(gene, n_frs1_snp, n_frs1_snp/n_sgrna_dict[gene])

C1_00220W 0 0.0
C7_02770W 0 0.0
C7_02060W 0 0.0
C7_01650W 0 0.0
C7_01180W 0 0.0
C7_00730W 0 0.0
C7_00210C 1 0.25
C6_04120C 0 0.0
C6_03110C 0 0.0
C6_02570C 2 0.5
C6_02370C 2 0.5
C6_01980C 2 0.5
C6_01280W 0 0.0
C6_00870C 1 0.25
C5_05410C 0 0.0
C5_05190W 0 0.0
C5_02900W 2 0.5
C5_01720C 1 0.25
C5_01460W 2 0.5
C5_01240W 0 0.0
C5_00790C 0 0.0
C5_00770C 2 0.5
C4_04820C 0 0.0
C4_04260C 2 0.5
C4_03330W 1 0.25
C4_03200C 0 0.0
C4_01980C 0 0.0
C4_01880W 0 0.0
C4_01870C 0 0.0
C4_01750C 0 0.0
C4_01150W 0 0.0
C7_02680W 0 0.0
C7_02950C 0 0.0
C4_00560C 0 0.0
C7_03000C 0 0.0
CR_10650W 0 0.0
CR_10010C 0 0.0
CR_09760W 0 0.0
CR_09320C 2 0.5
CR_07710W 0 0.0
CR_07390C 2 0.5
CR_06640C 0 0.0
CR_05100W 0 0.0
CR_04520W 2 0.5
CR_04140W 1 0.25
CR_03950W 0 0.0
CR_03930C 1 0.25
CR_03530W 1 0.25
CR_03430W 1 0.25
CR_02990C 2 0.5
CR_02580W 1 0.25
CR_02540W 0 0.0
CR_02090C 0 0.0
CR_01620C 0 0.0
CR_01370C 0 0.0
CR_01300W 0 0.0
CR_00890C 0 0.0
CR_00680W 0 0.0
CR_00510C 0 0.0
CR_00480W 0 0.0
CR_00410W 0 0.0
CR_00250W 0 0.0

In [ ]:
sgRNA_df[['Gene', 'alias', 'SgRNA', 'Barcode', 'PAM', 'control', 'strand', 'atg_dist', 'B_mismatch', 'dist_TSS', 'abs_TSS_dist', 'GC_content',
              'snp_fRS1', 'snp_hom_fRS1',
              'snp_fRS242', 'snp_hom_fRS242',
              'snp_fRS246', 'snp_hom_fRS246'              
              ]].to_csv('./sgRNA_annot.csv', sep=',')